# STEP 1: Overview – Script Description
# ------------------------------------
# This script automates updating ESP-r rotate.txt files using EPC data.
#
# What it does:
# 1. Loads the EPC dataset CSV file.
# 2. Creates a mapping between BUILDING_REFERENCE_NUMBER and ORIENTATION_ESPR_ROT.
# 3. Iterates through each building folder in the baseline_models directory.
# 4. Searches for any rotate.txt file within the building's model subfolders.
# 5. Replaces all instances of the string "ROTATE" with the corresponding orientation value.
# 6. Saves the updated file in place.
# 7. Skips buildings or files if data or rotate.txt is missing.
#
# Requirements:
# - Python 3
# - pandas
#
# Notes:
# - Safe to re-run (will only replace remaining "ROTATE" strings).
# - Prints progress and skipped cases for debugging.

In [ ]:
# STEP 2: Load EPC Data and Create Lookup Dictionary
# -------------------------------------------------

import pandas as pd
from pathlib import Path

# File paths
epc_path = Path("/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_NPV_IRR_update.csv")
baseline_models_path = Path("/Users/jakegehrung/Desktop/data_raw/baseline_models")

# Load dataset
df = pd.read_csv(epc_path)

# Ensure correct data types
df["BUILDING_REFERENCE_NUMBER"] = df["BUILDING_REFERENCE_NUMBER"].astype(str)

# Create lookup dictionary
orientation_lookup = dict(zip(
    df["BUILDING_REFERENCE_NUMBER"],
    df["ORIENTATION_ESPR_ROT"]
))

print(f"Loaded EPC data for {len(orientation_lookup)} buildings.")

Loaded EPC data for 117 buildings.


In [2]:
# STEP 3: Define Function to Update rotate.txt
# --------------------------------------------

def update_rotate_file(file_path, rotation_value):
    try:
        with open(file_path, "r") as f:
            content = f.read()
        
        # Replace ROTATE with actual value
        new_content = content.replace("ROTATE", str(rotation_value))
        
        # Only write if change occurred
        if content != new_content:
            with open(file_path, "w") as f:
                f.write(new_content)
            print(f"Updated: {file_path}")
        else:
            print(f"No change needed: {file_path}")
    
    except Exception as e:
        print(f"Error processing {file_path}: {e}")

In [3]:
# STEP 4: Iterate Through Buildings and Apply Updates
# --------------------------------------------------

processed = 0
skipped_no_data = 0
skipped_no_file = 0

for building_dir in baseline_models_path.iterdir():
    
    if not building_dir.is_dir():
        continue
    
    building_id = building_dir.name
    
    # Check if building exists in EPC dataset
    if building_id not in orientation_lookup:
        skipped_no_data += 1
        continue
    
    rotation_value = orientation_lookup[building_id]
    found_rotate_file = False
    
    # Search through model subfolders
    for subfolder in building_dir.iterdir():
        cfg_path = subfolder / "cfg" / "rotate.txt"
        
        if cfg_path.exists():
            found_rotate_file = True
            update_rotate_file(cfg_path, rotation_value)
    
    if found_rotate_file:
        processed += 1
    else:
        skipped_no_file += 1

print("\n--- Summary ---")
print(f"Processed buildings: {processed}")
print(f"Skipped (no EPC data): {skipped_no_data}")
print(f"Skipped (no rotate.txt): {skipped_no_file}")

Updated: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001664929/SemiDetached_2F/cfg/rotate.txt

--- Summary ---
Processed buildings: 1
Skipped (no EPC data): 0
Skipped (no rotate.txt): 116
